# 2 · Features, Models and Training

**Multimodal Recommender System for E-Commerce Item Cold-Start**

How products became numbers, what the eight systems are, the hyperparameters
they were trained with, and what the training logs show.

This notebook covers the **Features** and **Algorithm** sections of the report,
and the parameter sweeps required under **Results**.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = Path.cwd().parent
REPORTS = PROJECT / "reports"
TABLES = PROJECT / "results" / "tables"
FIGURES = PROJECT / "results" / "figures"
LOGS = PROJECT / "logs"
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "legend.frameon": False,
})

BLUE, ORANGE, GREEN, RED, PURPLE = "#4c72b0", "#dd8452", "#55a868", "#c44e52", "#8172b3"

DISPLAY = {
    "ncf": "NCF", "image_only": "Image only", "text_only": "Text only",
    "content_only": "Image + text", "interaction_image": "History + image",
    "interaction_text": "History + text", "concat_fusion": "Concat fusion",
    "attention_fusion": "Attention fusion",
}
ORDER = list(DISPLAY)


def read(name, folder=REPORTS):
    path = folder / name
    return pd.read_csv(path) if path.exists() else None


print("logs available:", len(list(LOGS.glob("*.csv"))))

## 1. Turning products into numbers

Two pretrained encoders convert each product into fixed-length vectors. Neither
is fine-tuned.

| modality | encoder | output | items encoded |
|---|---|---|---|
| image | CLIP ViT-B/32 | 512 dimensions | 9,479 |
| text | SBERT all-mpnet-base-v2 | 768 dimensions | 9,503 |

**Why the encoders are frozen**, in order of importance:

**Comparability.** Every system in the comparison sees identical image and text
features, so a difference between systems is attributable to the fusion
mechanism rather than to one having learned better features. Fine-tuning would
confound precisely the thing the experiment isolates.

**Statistics.** Fine-tuning CLIP would introduce roughly 87 million trainable
parameters against 93,701 training interactions — a ratio that invites
memorisation rather than generalisation.

**Feasibility.** CLIP, SBERT and the recommender would not fit together in 6 GB
of accelerator memory. Extracting once removes both encoders from the
training-time budget entirely; the recommender never loads them.

### Raw and derived features

The report asks for both. Nothing here is hand-engineered — the derived features
are encoder outputs and learned projections.

| feature | type | dimension | source |
|---|---|---|---|
| user identifier | raw | — | interaction log |
| product identifier (`parent_asin`) | raw | — | interaction log |
| interaction timestamp | raw | — | used for the chronological split |
| product title | raw text | — | item metadata |
| feature bullets | raw text | — | item metadata |
| product description | raw text | — | item metadata |
| product photograph | raw image | — | item metadata, MAIN variant |
| user embedding | derived, learned | 16 | trained from interactions |
| item embedding | derived, learned | 16 | trained from interactions |
| image embedding | derived, frozen | 512 | CLIP ViT-B/32 |
| text embedding | derived, frozen | 768 | SBERT all-mpnet-base-v2 |
| projected signal representations | derived, learned | 128 each | linear projection per modality |

Text is assembled as title, then feature bullets (capped at 400 characters),
then description (capped at 600). The order matters because the encoder
truncates beyond a token limit, so the ordering determines what survives. In
practice truncation never binds: the longest assembled text is 1,248 characters
against a limit near 1,500.

## 2. Do the embeddings carry information?

Embeddings can be present, correctly shaped and numerically valid while encoding
nothing useful. That failure produces no error.

The check: are two laptops more similar to each other than a laptop and a pair
of headphones? Two thousand pairs sampled within each domain, two thousand
across.

In [ ]:
similarity = read("feature_similarity.csv")
if similarity is not None:
    pivot = similarity.pivot(index="domain", columns="modality", values="margin")

    fig, ax = plt.subplots(figsize=(7, 3.2))
    positions = np.arange(len(pivot))
    ax.bar(positions - 0.19, pivot["image"], 0.38, label="image", color=ORANGE)
    ax.bar(positions + 0.19, pivot["text"], 0.38, label="text", color=BLUE)
    ax.set_xticks(positions)
    ax.set_xticklabels(pivot.index, rotation=20, ha="right")
    ax.set_ylabel("Within-domain minus cross-domain similarity")
    ax.set_title("Both encoders separate product domains; text does so more strongly")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.legend(fontsize=8)
    fig.savefig(FIGURES / "semantic_margins.png")
    plt.show()

    display(similarity.pivot(index="domain", columns="modality",
                             values=["within_domain", "cross_domain", "margin"]).round(3))
    print("mean margin  image:", round(pivot["image"].mean(), 3),
          " text:", round(pivot["text"].mean(), 3))

Within-domain similarity exceeds cross-domain similarity in **all ten
domain-modality combinations**. The multimodal premise is evidenced rather than
assumed.

**Text separates domains roughly 2.5 times more strongly than images** — a mean
margin of +0.257 against +0.102. A title states the category outright, whereas
photographs of a laptop and a tablet are genuinely similar.

This was recorded as a prediction before any model was trained, and it held:
text-based systems outperform image-based ones at cold-start.

Image cross-domain similarity sits high, around 0.62, which is characteristic of
CLIP on e-commerce photography. White backgrounds and studio lighting make all
product images somewhat alike, compressing the range within which discrimination
must happen.

### A scale mismatch that had to be corrected

| modality | mean vector norm |
|---|---|
| image (CLIP) | 10.331 |
| text (SBERT) | 1.000 |

`all-mpnet-base-v2` applies L2 normalisation inside its own pipeline, so its
outputs are unit vectors regardless of what is requested. CLIP applies none.

A linear layer weights its inputs by magnitude, so without correction the image
signal would dominate the fused representation from initialisation. The
methodological damage exceeds the optimisation damage: the ablation study could
not then distinguish *text contributes less information* from *text vectors are
ten times smaller*.

**A `LayerNorm` is applied per modality before projection**, identically across
every variant so the comparison remains controlled.

## 3. The eight systems

Every system is the same class with different switches. Three signals —
collaborative, image, text — each on or off, plus a fusion mode.

Two systems sharing one implementation cannot differ through incidental
engineering choices, only through the architectural choice under test. Eight
separate files would differ in a dozen unintended ways, each an invisible
confound.

In [ ]:
systems = pd.DataFrame([
    ("ncf",               "yes", "—",   "—",   "concat",    1_679_041),
    ("image_only",        "—",   "yes", "—",   "concat",    1_129_217),
    ("text_only",         "—",   "—",   "yes", "concat",    1_162_497),
    ("content_only",      "—",   "yes", "yes", "concat",    1_245_697),
    ("interaction_image", "yes", "yes", "—",   "concat",            0),
    ("interaction_text",  "yes", "—",   "yes", "concat",            0),
    ("concat_fusion",     "yes", "yes", "yes", "concat",    1_878_721),
    ("attention_fusion",  "yes", "yes", "yes", "attention", 1_911_617),
], columns=["system", "collaborative", "image", "text", "fusion", "parameters"])
systems["display"] = systems["system"].map(DISPLAY)
systems[["display", "collaborative", "image", "text", "fusion"]]

The first six form the headline comparison. The two mixing the collaborative
signal with a single content modality complete the **seven-way structural
ablation**, which requires every non-empty combination of the three signals.

**Parameter counts are dominated by embedding tables**, so capacity cannot
explain differences between the fusion designs: concatenation and attention
differ by 1.8%.

### How a new product is handled

A withheld item has no learned collaborative representation. Both naive
responses are wrong.

A **zero vector** is not a meaningful point in the learned space — the origin is
where an untrained embedding sits, and the model reads it as a product with
particular properties rather than as an absence. The **mean item vector** is
worse: it is a constant, so every cold-start item receives an identical
collaborative contribution.

The model therefore **branches**. Under attention, the missing signal is excluded
from the attention keys, so its weight is exactly zero. Under concatenation, its
slot receives a learned parameter that the model observes during training and
learns to interpret as absence.

The same mechanism serves three purposes: withheld items, the 24 items without a
retrievable image, and the deliberate suppression used for the Blind and Silent
evaluation conditions. One mechanism, no special cases — which is what keeps
those three sets of results comparable.

### Verifying the branch is real

A model can produce entirely plausible cold-start scores while silently
consuming zeros. The test scores a withheld item twice: once through the
content-only path, and once with its collaborative embedding explicitly zeroed.
**Identical results would prove the branching is fake.**

| system | cold-start vs warm path | cold-start vs zero substitution |
|---|---|---|
| concat_fusion | 0.012 | **0.007** |
| attention_fusion | 0.234 | **0.234** |

Both non-zero. Attention diverges more because masking removes a key from the
softmax entirely, redistributing its whole weight.

Untrained attention weights on a withheld item read
`[0.0000, 0.5007, 0.4993]` — the collaborative channel receives *exactly* zero
and its mass moves to the content signals.

## 4. Hyperparameters

The report requires the final model described in full.

In [ ]:
hyper = pd.DataFrame([
    ("Architecture", "embedding_dim", 16, "user and item embedding width"),
    ("Architecture", "fusion_dim", 128, "shared projection width per signal"),
    ("Architecture", "attention_heads", 4, "32 dimensions per head"),
    ("Architecture", "mlp_layers", "[128, 64, 32]", "scoring head"),
    ("Architecture", "dropout", 0.2, "applied in the scoring head"),
    ("Architecture", "modality_normalisation", "layernorm", "per modality, before projection"),
    ("Architecture", "collaborative_dropout", 0.2, "trains the content-only path"),
    ("Optimisation", "optimiser", "Adam", ""),
    ("Optimisation", "learning_rate", 0.001, ""),
    ("Optimisation", "weight_decay", 1e-05, ""),
    ("Optimisation", "batch_size", 256, ""),
    ("Optimisation", "num_negatives", 4, "negatives sampled per positive"),
    ("Optimisation", "loss", "pairwise BPR", "optimises order, not absolute score"),
    ("Optimisation", "max_epochs", 100, ""),
    ("Optimisation", "early_stopping_patience", 20, "on validation NDCG@10"),
    ("Protocol", "split_seed", 42, "frozen and checksummed"),
    ("Protocol", "development_seed", 1337, "all tuning"),
    ("Protocol", "statistical_seeds", "1, 2, 3, 4, 5", "locked configuration only"),
    ("Feature", "clip_model", "openai/clip-vit-base-patch32", "frozen"),
    ("Feature", "sbert_model", "all-mpnet-base-v2", "frozen"),
], columns=["group", "parameter", "value", "note"])
hyper

### Two additions beyond a standard implementation

**`collaborative_dropout` at 0.2.** Every item present in training has a
collaborative representation by definition, so a model trained without
intervention *never once encounters* the condition it must handle at cold-start
inference. The content-only path would be correctly implemented and still
produce out-of-distribution scores. Suppressing the collaborative signal on a
random fifth of samples makes that path a trained path, exercised roughly 18,700
times per epoch.

This is a documented addition, not something a build manual prescribed, and it
was verified empirically rather than argued for — see section 6.

**`modality_normalisation`.** The answer to the 10.3 : 1 scale mismatch, applied
identically across all variants so it cannot become a confound.

### One parameter that cannot learn

The final layer's bias receives **exactly zero gradient**. This is correct by
construction: the pairwise objective depends on the *difference* between two
scores for the same user, so a constant present in both cancels. The parameter is
mathematically unidentifiable rather than broken, and is retained because it
becomes identifiable under a pointwise loss.

## 5. What the training logs show

Every configuration was trained across five statistical seeds — forty runs in
total, plus the tuning runs. The logs record loss, validation NDCG@10 and item
coverage at every epoch.

In [ ]:
frames = []
for path in sorted(LOGS.glob("*_seed*.csv")):
    name = path.stem
    if any(tag in name for tag in ("cd", "ed")):
        continue
    system, _, seed = name.rpartition("_seed")
    if system in DISPLAY:
        frame = pd.read_csv(path)
        frame["system"], frame["seed"] = system, int(seed)
        frames.append(frame)

logs = pd.concat(frames, ignore_index=True) if frames else None
if logs is not None:
    print(f"{len(logs):,} epochs logged across "
          f"{logs.groupby(['system', 'seed']).ngroups} runs")
    display(logs.groupby("system")
            .agg(runs=("seed", "nunique"), epochs=("epoch", "max"),
                 best_ndcg=("val_ndcg", "max"))
            .reindex(ORDER).round(4))

In [ ]:
if logs is not None:
    statistical = logs[logs["seed"].isin([1, 2, 3, 4, 5])]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))

    for system, colour in zip(["ncf", "concat_fusion", "attention_fusion"],
                              [RED, GREEN, PURPLE]):
        subset = statistical[statistical["system"] == system]
        mean = subset.groupby("epoch")["train_loss"].mean()
        axes[0].plot(mean.index, mean.values, label=DISPLAY[system],
                     color=colour, linewidth=1.4)
    axes[0].axhline(np.log(2), color="black", linestyle="--", linewidth=0.9, alpha=0.6)
    axes[0].text(1, np.log(2) * 1.02, "chance (log 2)", fontsize=7.5, alpha=0.7)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Training loss")
    axes[0].set_title("Training loss"); axes[0].legend(fontsize=7.5)

    for system, colour in zip(["ncf", "concat_fusion", "attention_fusion"],
                              [RED, GREEN, PURPLE]):
        subset = statistical[statistical["system"] == system]
        mean = subset.groupby("epoch")["val_ndcg"].mean()
        axes[1].plot(mean.index, mean.values, label=DISPLAY[system],
                     color=colour, linewidth=1.4)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation NDCG@10")
    axes[1].set_title("Validation performance"); axes[1].legend(fontsize=7.5)

    if "coverage" in statistical.columns:
        for system, colour in zip(["ncf", "concat_fusion", "attention_fusion"],
                                  [RED, GREEN, PURPLE]):
            subset = statistical[statistical["system"] == system]
            mean = subset.groupby("epoch")["coverage"].mean()
            axes[2].plot(mean.index, mean.values, label=DISPLAY[system],
                         color=colour, linewidth=1.4)
        axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Distinct items recommended")
        axes[2].set_title("Item coverage")
        axes[2].legend(fontsize=7.5)

    fig.tight_layout()
    fig.savefig(FIGURES / "training_curves.png")
    plt.show()

**Training loss falls well below chance.** A model ranking a positive and a
negative item with equal confidence scores exactly log 2 ≈ 0.693, so the dashed
line marks a system that has learned nothing.

**Validation performance plateaus long before training loss does**, which is why
early stopping retains the best checkpoint rather than the last. In the first
NCF run, loss fell from 0.59 to 0.20 while validation NDCG drifted downward after
epoch seven.

**Item coverage** counts how many distinct products appear in any user's top ten.
It is not an accuracy measure; it detects a failure accuracy conceals — a model
that has collapsed onto recommending broadly the same popular items to everyone.
It earned its place, as section 6 shows.

In [ ]:
if logs is not None:
    statistical = logs[logs["seed"].isin([1, 2, 3, 4, 5])]
    fig, axes = plt.subplots(2, 4, figsize=(13, 5.6), sharex=True)
    for ax, system in zip(axes.ravel(), ORDER):
        subset = statistical[statistical["system"] == system]
        for seed in sorted(subset["seed"].unique()):
            run = subset[subset["seed"] == seed]
            ax.plot(run["epoch"], run["val_ndcg"], linewidth=0.9, alpha=0.8)
        ax.set_title(DISPLAY[system], fontsize=9)
        ax.set_ylim(0.012, 0.032)
    for ax in axes[1]:
        ax.set_xlabel("Epoch")
    for ax in axes[:, 0]:
        ax.set_ylabel("Validation NDCG@10")
    fig.suptitle("Validation trajectories, five seeds per system", fontsize=10)
    fig.tight_layout()
    fig.savefig(FIGURES / "seed_trajectories.png")
    plt.show()

In [ ]:
seed_summary = read("seed_summary.csv")
if seed_summary is not None:
    seed_summary["display"] = seed_summary["system"].map(DISPLAY).fillna(
        seed_summary["system"])
    table = seed_summary.sort_values("ndcg_mean", ascending=False)

    fig, ax = plt.subplots(figsize=(7, 3.4))
    positions = np.arange(len(table))
    ax.barh(positions, table["ndcg_mean"], xerr=table["ndcg_std"],
            color=BLUE, height=0.62, error_kw={"elinewidth": 1, "capsize": 3})
    ax.set_yticks(positions)
    ax.set_yticklabels(table["display"])
    ax.invert_yaxis()
    ax.set_xlabel("Validation NDCG@10 (mean ± 1 s.d., five seeds)")
    ax.set_title("Validation performance across seeds")
    fig.savefig(FIGURES / "seed_summary.png")
    plt.show()

    display(table[["display", "runs", "ndcg_mean", "ndcg_std",
                   "coverage_mean", "epoch_mean"]].round(4)
            .rename(columns={"display": "system"}))

**The spread is narrow relative to the differences between systems.** Best to
worst spans 0.0026 while the typical standard deviation is 0.0018 — roughly 1.4
standard deviations across eight architecturally different systems.

On warm-start validation these systems are largely indistinguishable. That is a
property of the regime: ranking one held-out item among 7,602 candidates on a
matrix with roughly twelve interactions per item is hard for any architecture.
The cold-start condition is where they separate, and that is notebook 3.

## 6. Parameter sweeps

The report asks for performance graphs from parameter sweeps. Two were run.

### The problem that prompted them

The first round of training produced a result that could not be true:
`image_only` scored 4.5 times every other system, **including `content_only`,
which receives images and text**. A model given strictly more information cannot
legitimately perform worse, because it could learn to ignore the addition.

The coverage metric exposed a second problem the accuracy column concealed:
`attention_fusion` was placing just **29 distinct items out of 7,602** into any
user's top ten, with **92% of users receiving the same product**. The popularity
baseline gives 13 items and 98.6%. The flagship model was behaving almost
identically to a non-learned baseline.

In [ ]:
diagnostic = read("diagnostic_coverage.csv")
if diagnostic is not None:
    fig, ax = plt.subplots(figsize=(7, 3.4))
    table = diagnostic.sort_values("coverage")
    colours = ["#9e9e9e" if s in ("popularity", "random") else BLUE
               for s in table["system"]]
    positions = np.arange(len(table))
    ax.barh(positions, table["distinct_items"], color=colours, height=0.6)
    ax.set_yticks(positions)
    ax.set_yticklabels([DISPLAY.get(s, s) for s in table["system"]])
    ax.set_xlabel(f"Distinct items recommended, of {table['candidates'].iloc[0]:,}")
    ax.set_title("Item coverage before tuning: most systems had collapsed")
    for y, (n, share) in enumerate(zip(table["distinct_items"],
                                       table["most_recommended_item_share"])):
        ax.text(n + 20, y, f"{n:,}  ({share:.0%} of users get the top item)",
                va="center", fontsize=7.5)
    ax.set_xlim(0, table["distinct_items"].max() * 1.9)
    fig.savefig(FIGURES / "coverage_diagnostic.png")
    plt.show()
    display(diagnostic)

**Accuracy metrics cannot distinguish a personalised recommender from one that
has collapsed onto popularity.** Item Coverage@10 was specified in the
configuration before any model existed, precisely for this reason, and it earned
its place. It was subsequently added to the per-epoch training log so a collapse
is visible from the first epoch rather than found by a separate investigation.

In [ ]:
tuning = read("tuning_sweep.csv")
if tuning is not None:
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))

    for dimension, colour, marker in zip([64, 16], [BLUE, GREEN], ["o", "s"]):
        subset = tuning[tuning["embedding_dim"] == dimension].sort_values(
            "collaborative_dropout")
        axes[0].plot(subset["collaborative_dropout"], subset["best_val_ndcg"],
                     marker=marker, color=colour, linewidth=1.5,
                     label=f"embedding {dimension}")
        axes[1].plot(subset["collaborative_dropout"], subset["coverage_at_best"],
                     marker=marker, color=colour, linewidth=1.5,
                     label=f"embedding {dimension}")

    axes[0].set_xlabel("Collaborative dropout")
    axes[0].set_ylabel("Validation NDCG@10")
    axes[0].set_title("Accuracy")
    axes[0].legend(fontsize=8)

    axes[1].set_xlabel("Collaborative dropout")
    axes[1].set_ylabel("Distinct items recommended")
    axes[1].set_title("Coverage")
    axes[1].legend(fontsize=8)

    fig.suptitle("Parameter sweep: collaborative dropout by embedding width",
                 fontsize=10)
    fig.tight_layout()
    fig.savefig(FIGURES / "parameter_sweep.png")
    plt.show()
    display(tuning.sort_values("best_val_ndcg", ascending=False))

### What the sweep established

**A hypothesis was refuted.** The suspicion was that the collaborative signal,
present on most training samples and easy to memorise, was starving the content
pathways of gradient. If so, suppressing it further should have helped.
It did not: results degraded monotonically in both accuracy and coverage as
suppression increased. The collaborative pathway was not crowding out content;
removing it removed real signal.

**A different finding emerged.** Narrowing the embedding from 64 to 16
dimensions improved both accuracy and coverage. With roughly twelve interactions
per item, a 64-dimensional embedding is under-determined by about five to one, so
most fitted dimensions are unconstrained noise. Sixteen brings the ratio closer
to one.

**An internal consistency check fell out of it.** At dropout 1.0 with embedding
64 the model scored 0.0234 while `content_only` scored 0.0238. Suppressing the
collaborative signal on *every* sample should make a fusion model equivalent to a
content-only model, and it did.

### Verifying collaborative dropout

The parameter was tested rather than argued for:

| collaborative_dropout | validation NDCG@10 | coverage |
|---|---|---|
| 0.0 | 0.0264 | 194 |
| **0.2** | **0.0290** | **335** |

Both improve at 0.2 and degrade beyond it — the shape of a genuine trade-off
rather than a free lunch.

### Regression to the mean

The tuned configuration was selected on seed 1337. When all systems were then
retrained across the five statistical seeds, the numbers fell:

| system | seed 1337 | five-seed mean |
|---|---|---|
| concat_fusion | 0.0301 | 0.0244 ± 0.0018 |
| attention_fusion | 0.0290 | 0.0243 ± 0.0022 |

`concat_fusion` on seed 1337 sits **3.2 standard deviations above its own
five-seed mean**. This is expected and important: a configuration selected
because it performed well on one seed has optimistically biased performance on
that seed.

The `image_only` anomaly resolved the same way, falling from 0.1131 to 0.0258.
Its original trajectory swung six-fold between consecutive epochs on a *fixed*
validation set, and early stopping — which takes the maximum over epochs — is a
biased estimator that selects for upward fluctuation.

**This is why every figure in notebook 3 reports a mean with a spread rather
than a best run.**

## 7. Feature importance

The report asks for an importance ranking. For a learned multimodal model there
is no coefficient to read, so importance is measured three independent ways.

**Modality suppression** removes a signal from a fully trained model and measures
the drop. **Attention weights** report how much of the fused representation each
signal contributes. **Neighbour co-purchase lift** asks whether items that are
close in a modality's embedding space are bought by the same people — measured
directly on the embeddings, independent of any model.

In [ ]:
importance = pd.DataFrame([
    ("Collaborative (purchase history)", 70.1, "—", "highest on established items, absent on new ones"),
    ("Text (title, features, description)", 15.7, "6.4x", "strongest content signal; transfers best"),
    ("Image (product photograph)", 14.2, "4.1x", "real but weaker signal; transfers poorly"),
], columns=["signal", "attention weight on warm items (%)",
            "co-purchase lift over random", "note"])
display(importance)

neighbour = read("neighbour_copurchase.csv")
if neighbour is not None:
    display(neighbour)

reliance = read("modality_reliance.csv")
if reliance is not None:
    pivot = reliance.pivot(index="signal", columns="condition", values="mean_weight")
    fig, ax = plt.subplots(figsize=(6.5, 3))
    positions = np.arange(len(pivot))
    ax.bar(positions - 0.19, pivot["warm"], 0.38, label="established items", color=BLUE)
    ax.bar(positions + 0.19, pivot["cold_start"], 0.38, label="new items", color=GREEN)
    ax.set_xticks(positions); ax.set_xticklabels(pivot.index)
    ax.set_ylabel("Mean attention weight")
    ax.set_title("Attention reallocates when purchase history is unavailable")
    ax.legend(fontsize=8)
    fig.savefig(FIGURES / "attention_reallocation.png")
    plt.show()
    display(pivot.round(4))

**The attention figure is the interpretability evidence the project's objectives
require.** On established items the model places roughly 70% of its attention on
purchase history. On new items that weight is **exactly zero** — the masking
works — and its mass reallocates to image and text.

**A caution about that 70%.** It is measured on established items, where the
collaborative signal exists. It does not mean the content pathways are
unimportant; it means the model relies on the easiest available signal when that
signal is present. Notebook 3 shows the consequence: the fusion model that
relies most heavily on collaborative history is also the *weakest* on new items,
because its content pathways developed least.

**Text ranks above image on every measure** — attention weight, co-purchase
lift, semantic margin, and cold-start performance. Four independent
measurements, one ordering.

## Summary for the report

**Features.** Two frozen encoders produce 512-dimensional image vectors and
768-dimensional text vectors from product photographs and metadata. Both were
validated to encode real semantic structure before use. A 10.3 : 1 scale
mismatch between them is corrected by per-modality normalisation.

**Algorithm.** A single implementation with three switchable signals and two
fusion modes, giving eight systems and the complete seven-way structural
ablation. Cold-start items take a genuine computational branch rather than
receiving substituted zeros, and the branch is verified by differential test.

**Training.** Pairwise ranking objective, four negatives per positive drawn only
from representable items, Adam, early stopping on validation NDCG@10 with the
best checkpoint retained. Forty runs across five statistical seeds.

**Tuning.** Two parameters swept over eight configurations, selected on both
accuracy and coverage. One hypothesis refuted in the process, and one
methodological finding recorded: single-seed selection produced results up to
3.2 standard deviations above the multi-seed mean.